# PINN求解方程

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import time
import scipy.io as io
import matplotlib.pyplot as plt


base_dir = os.getcwd()
print(f"Current working directory: {base_dir}")
# base_dir = os.path.dirname(os.path.abspath(__file__))
output_path = os.path.join(base_dir, 'Hom2', 'Output', 'ode_pred.mat')

np.random.seed(1234)
torch.manual_seed(1234)

#size of the DNN
layers = [1] + 2*[30] + [1]

## DNN网络

In [ ]:
class DNN(nn.Module):
    def __init__(self, layers):
        super(DNN, self).__init__()
        self.layers = layers
        self.linears = nn.ModuleList()
        for i in range(1, len(layers)):
            in_dim = layers[i-1]
            out_dim = layers[i]
            linear = nn.Linear(in_dim, out_dim)
            std = np.sqrt(2 / (in_dim + out_dim))
            nn.init.normal_(linear.weight, mean=0., std=std)
            nn.init.zeros_(linear.bias)
            self.linears.append(linear)

    def forward(self, x):
        a = x
        for i in range(len(self.linears)-1):
            a = torch.tanh(self.linears[i](a))
        y = self.linears[-1](a)
        return y

    def pdenn(self, x):
        x = x.clone().requires_grad_(True)
        u = self.forward(x)
        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
        f = torch.sin(x) + 2*torch.sin(2*x) + 3*torch.sin(3*x) + 4*torch.sin(4*x) + 8*torch.sin(8*x)
        r = f + u_xx
        f_pred = -u_xx
        return u, f_pred, r

In [3]:
def exact_u_sol(x):
    u = x + np.sin(x) + np.sin(2*x)/2 + np.sin(3*x)/3 + np.sin(4*x)/4 + np.sin(8*x)/8
    return u

def exact_f_sol(x):
    f = np.sin(x) + 2*np.sin(2*x) + 3*np.sin(3*x) + 4*np.sin(4*x) + 8*np.sin(8*x)
    return f

def training_data():
    xmin, xmax = -np.pi, np.pi
    num_f_train = 20
    x_f_train = np.linspace(xmin, xmax, num_f_train).reshape((-1, 1))
    f_train = exact_f_sol(x_f_train)
    x_bc_l = np.array([xmin]).reshape((-1, 1))
    x_bc_r = np.array([xmax]).reshape((-1, 1))
    u_l, u_r = exact_u_sol(x_bc_l), exact_u_sol(x_bc_r)
    x_u_train = np.vstack((x_bc_l, x_bc_r))
    u_train = np.vstack((u_l, u_r))

    return x_u_train, u_train, x_f_train, f_train

def add_data(x_r_test, err_current):
    x_id = np.argmax(np.absolute(err_current))
    x_add = x_r_test[x_id]
    return x_add

def build_dataset(x_current, x_add):
    x_add = np.reshape(x_add, (-1, 1))
    x_f_batch = np.vstack((x_current, x_add))
    return x_f_batch

In [4]:
def main():
    # generating training data
    x_u_train, u_train, x_f_train, f_train = training_data()

    x_u_train_t = torch.tensor(x_u_train, dtype=torch.float32)
    u_train_t = torch.tensor(u_train, dtype=torch.float32)

    # physics-infromed neural networks
    model = DNN(layers)
    optimizer = torch.optim.Adam(model.parameters(), lr=1.0e-3)

    n = 0
    nmax = 30
    err = 1.

    # loop for residual
    n_r_max = 20000

    start_time = time.perf_counter()
    while n <= nmax and err > 1.0e-2:
        print('Loop: %d'%(n+1))
        if n == 0:
            x_f_batch = x_f_train
        else:
            x_f_batch = build_dataset(x_f_batch, x_add)
        
        x_f_batch_t = torch.tensor(x_f_batch, dtype=torch.float32)
        
        n_r = 0
        loss_r_ = 1.0
        while n_r <= n_r_max and loss_r_ >= 1.0e-6: 
            optimizer.zero_grad()
            
            u_pred = model(x_u_train_t)
            _, _, r_pred = model.pdenn(x_f_batch_t)
            
            # loss for bcs
            loss_u = torch.mean((u_pred - u_train_t)**2)
            # loss for residual
            loss_r = torch.mean(r_pred**2)
            loss = loss_u + loss_r
            
            loss.backward()
            optimizer.step()
            
            loss_r_ = loss.item()
            n_r += 1
            if n_r%100 == 0:
                print('Steps: %d, Loss: %.3e'%(n_r, loss_r_))
        n += 1
        
        print('# of training data: %d'%(x_f_batch.shape[0]))
        x_r_test = 2*np.pi*np.random.rand(100, 1) - np.pi
        x_r_test_t = torch.tensor(x_r_test, dtype=torch.float32)
        
        _, f_test, r_test = model.pdenn(x_r_test_t)
        f_test = f_test.detach().numpy()
        r_test = r_test.detach().numpy()
        
        err = np.mean(np.absolute(r_test))
        print('Mean error of residual: %.5f'%(err))
        x_add = add_data(x_r_test, r_test)
        print('Added points is %.3f'%(x_add))

    stop_time = time.perf_counter()
    print('Duration time is %.3f seconds'%(stop_time - start_time))

    _, f_train_batch, _ = model.pdenn(torch.tensor(x_f_batch, dtype=torch.float32))
    f_train_batch = f_train_batch.detach().numpy()
    
    num_test = 1001
    x_test = np.linspace(-np.pi, np.pi, num_test).reshape(-1, 1)
    x_test_t = torch.tensor(x_test, dtype=torch.float32)
    
    u_test = model(x_test_t).detach().numpy()
    _, f_test, _ = model.pdenn(x_test_t)
    f_test = f_test.detach().numpy()
    
    save_dict = {'x_u_train': x_u_train, 'u_train': u_train, 'x_f_train': x_f_train, 'f_train': f_train, 'x_test': x_test, \
                 'u_test': u_test, 'f_test': f_test}
    
    io.savemat(output_path, save_dict)

    u_ref, f_ref = exact_u_sol(x_test), exact_f_sol(x_test)

    err_u, err_f = np.linalg.norm(u_test - u_ref)/np.linalg.norm(u_ref), np.linalg.norm(f_test - f_ref)/np.linalg.norm(f_ref)
    print('Relative l2 errors: u: %3e, f: %.3e'%(err_u, err_f))

    plt.figure()
    plt.plot(x_u_train, u_train, 'bo')
    plt.plot(x_test, u_ref, 'k-')
    plt.plot(x_test, u_test, 'r--')
    plt.xlabel('x')
    plt.ylabel('u')
    plt.show()

    plt.figure()
    plt.plot(x_f_train, f_train, 'bo')
    plt.plot(x_f_batch, f_train_batch, 'b*')
    plt.plot(x_test, f_ref, 'k-')
    plt.plot(x_test, f_test, 'r--')
    plt.xlabel('x')
    plt.ylabel('f')
    plt.show()

if __name__ == '__main__':
    main()

Loop: 1
Steps: 100, Loss: 4.094e+01
Steps: 200, Loss: 2.813e+01
Steps: 300, Loss: 2.728e+01
Steps: 400, Loss: 2.472e+01
Steps: 500, Loss: 2.248e+01
Steps: 600, Loss: 2.029e+01
Steps: 700, Loss: 1.570e+01
Steps: 800, Loss: 1.151e+01


KeyboardInterrupt: 